# Kaggle Stage 2: PTv3 Baselines & Area 5 Evaluation
Trains baseline PTv3 models (Rows A, B, D) and evaluates Area 5 (Rows A, B, C, E, G).

**Environment:** Headless Kaggle GPU (T4 or T4 x2). Ephemeral scratch in `/tmp/workspace`.


In [ ]:
# 1. Environment diagnostics, repository checkout & dependencies
import os, sys, shutil, subprocess
from pathlib import Path

print('Python:', sys.version.split()[0])
!nvidia-smi

shm = shutil.disk_usage('/dev/shm')
print(f'/dev/shm capacity: {shm.total / (1024**2):.1f} MB (used: {shm.used / (1024**2):.1f} MB)')
work_disk = shutil.disk_usage('/kaggle/working' if Path('/kaggle').is_dir() else '.')
print(f'Working disk free: {work_disk.free / (1024**3):.2f} GB')

TARGET_DIR = Path('/tmp/point-cloud-upsampling')
REPO_URL = 'https://github.com/duckydodo/point-cloud-upsampling.git'

if TARGET_DIR.exists():
    print(f'Updating repository in {TARGET_DIR}...')
    !git -C {TARGET_DIR} pull --ff-only
else:
    print(f'Cloning repository into {TARGET_DIR}...')
    !git clone -q {REPO_URL} {TARGET_DIR}

%cd {TARGET_DIR}
if str(TARGET_DIR) not in sys.path:
    sys.path.insert(0, str(TARGET_DIR))

!pip install -q -r requirements.txt


In [ ]:
# 2. Execute Stage 2 Pipeline
# Toggle SMOKE_TEST: set to True for a fast 2-minute validation on synthetic data
SMOKE_TEST = False

smoke_flag = '--smoke-test' if SMOKE_TEST else ''
cmd = f'python kaggle/stage2_baselines_and_eval.py {smoke_flag}'.strip()
print(f'Running: {cmd}')
!{cmd}


In [ ]:
# 3. Artifact summary and disk usage in /kaggle/working
import os
from pathlib import Path

out_dir = Path('/kaggle/working') if Path('/kaggle').is_dir() else Path('kaggle_output')
print(f'=== Output Artifacts in {out_dir} ===')
total_bytes = 0
if out_dir.exists():
    for root, dirs, files in os.walk(out_dir):
        for f in sorted(files):
            p = Path(root) / f
            size_mb = p.stat().st_size / (1024**2)
            total_bytes += p.stat().st_size
            rel = p.relative_to(out_dir)
            print(f'  {rel} ({size_mb:.2f} MB)')
print(f'Total output size: {total_bytes / (1024**2):.2f} MB ({total_bytes / (1024**3):.3f} GB)')
print(f'Kaggle 20 GB persistence cap headroom: {20.0 - total_bytes / (1024**3):.2f} GB remaining.')
